In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

HOST = "https://www.ebi.ac.uk"
ACTIVITY_URL = f"{HOST}/chembl/api/data/activity.json"
TARGET_ID = "CHEMBL203"
PAGE_SIZE = 1000
TIMEOUT_SECONDS = 60
FIELDS = [
    "activity_id", "molecule_chembl_id", "canonical_smiles",
    "assay_chembl_id", "assay_description", "assay_type",
    "assay_variant_mutation", "bao_label", "target_organism",
    "standard_relation", "standard_units", "pchembl_value",
    "potential_duplicate", "data_validity_comment",
]

def get_json(url):
    with urlopen(url, timeout=TIMEOUT_SECONDS) as response:
        return json.load(response)


In [ ]:
params = {"target_chembl_id": TARGET_ID, "standard_type": "IC50", "limit": PAGE_SIZE}
first_page = get_json(f"{ACTIVITY_URL}?{urlencode(params)}")
expected_total = first_page["page_meta"]["total_count"]
records = [{field: activity.get(field) for field in FIELDS}
           for activity in first_page["activities"]]

def fetch_page(offset):
    url = f"{ACTIVITY_URL}?{urlencode(params | {'offset': offset})}"
    return get_json(url)

# Fetch independent pages concurrently; preserve their offset order in the result.
with ThreadPoolExecutor(max_workers=5) as pool:
    for page in pool.map(fetch_page, range(PAGE_SIZE, expected_total, PAGE_SIZE)):
        records.extend({field: activity.get(field) for field in FIELDS}
                       for activity in page["activities"])
if len(records) != expected_total or len({r["activity_id"] for r in records}) != expected_total:
    raise RuntimeError(f"Incomplete/overlapping ChEMBL download: {len(records)} of {expected_total} records")
raw = pd.DataFrame.from_records(records, columns=FIELDS)
print(f"EGFR IC50: {len(raw):,} raw activities; {raw['assay_chembl_id'].nunique():,} assays; "
      f"{raw['molecule_chembl_id'].nunique():,} ChEMBL molecule IDs")
display(raw[["molecule_chembl_id", "canonical_smiles", "pchembl_value",
             "assay_chembl_id", "assay_variant_mutation", "bao_label"]].head(5))


In [ ]:
pic50 = pd.to_numeric(raw["pchembl_value"], errors="coerce")
valid = (
    raw["standard_relation"].eq("=")
    & raw["standard_units"].eq("nM")
    & raw["potential_duplicate"].eq(0)
    & raw["data_validity_comment"].isna()
    & raw["canonical_smiles"].notna()
    & pic50.notna()
)
quality = raw.loc[valid].copy()
quality["pIC50"] = pic50.loc[valid]
print(f"Basic-quality subset: {len(quality):,} activities; "
      f"{quality['assay_chembl_id'].nunique():,} assays; "
      f"{quality['molecule_chembl_id'].nunique():,} ChEMBL molecule IDs")
display(quality[["molecule_chembl_id", "pIC50", "assay_chembl_id",
                 "assay_type", "bao_label", "assay_variant_mutation"]].head(5))


In [ ]:
for field in ("assay_type", "bao_label", "assay_variant_mutation"):
    print("\n" + field + " (activity rows):")
    display(quality[field].fillna("(not annotated)").value_counts().head(12)
            .rename_axis(field).to_frame("rows"))

candidate = quality.loc[
    quality["assay_type"].eq("B")
    & quality["bao_label"].eq("single protein format")
    & quality["assay_variant_mutation"].isna()
].copy()
print("\nB + single protein + no mutation annotation (NOT verified WT):")
print(f"{len(candidate):,} activities; {candidate['assay_chembl_id'].nunique():,} assays; "
      f"{candidate['molecule_chembl_id'].nunique():,} molecule IDs")


In [ ]:
assays = (candidate.groupby("assay_chembl_id")
          .agg(rows=("activity_id", "size"),
               molecule_ids=("molecule_chembl_id", "nunique"),
               description=("assay_description", "first"))
          .sort_values("rows", ascending=False))
assays["description_excerpt"] = assays["description"].fillna("").str.slice(0, 180)
assays["assay_url"] = [f"{HOST}/chembl/api/data/assay/{aid}.json" for aid in assays.index]
display(assays[["rows", "molecule_ids", "description_excerpt", "assay_url"]].head(15))

inspect_ids = list(dict.fromkeys([*assays.head(5).index, "CHEMBL5738142"]))
assay_details = {aid: get_json(f"{HOST}/chembl/api/data/assay/{aid}.json") for aid in inspect_ids}
for aid, detail in assay_details.items():
    print(f"\n{aid}: {assays.loc[aid, 'rows'] if aid in assays.index else '?'} candidate rows; "
          f"format={detail.get('bao_label')}; variant={detail.get('variant_sequence')}")
    print((detail.get("description") or "(no description)")[:520])


In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.info")

largest_fragment = rdMolStandardize.LargestFragmentChooser()

def standardize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(largest_fragment.choose(mol))

candidate["smiles"] = candidate["canonical_smiles"].map(standardize_smiles)
candidate = candidate.dropna(subset=["smiles"])
by_structure = candidate.groupby("smiles").agg(
    rows=("activity_id", "size"),
    assays=("assay_chembl_id", "nunique"),
    pic50_min=("pIC50", "min"),
    pic50_max=("pIC50", "max"),
)
by_structure["pic50_spread"] = by_structure["pic50_max"] - by_structure["pic50_min"]
across_assays = by_structure.loc[by_structure["assays"] > 1]
print(f"Normalized structures in candidate: {len(by_structure):,}")
print(f"Structures measured in multiple assays: {len(across_assays):,}")
print(f"Those with pIC50 spread >= 1: {int(across_assays['pic50_spread'].ge(1).sum()):,}")
display(across_assays.sort_values("pic50_spread", ascending=False).head(5))
